# Étape 1 — Analyse exploratoire et prétraitement

ERA5-Land, moyennes mensuelles, Algérie, 2015 → dernière année complète.

1. réduction de la résolution spatiale d'un facteur 4 (moyenne par blocs 4×4) ;
2. valeurs manquantes : les points hors terre d'ERA5-Land ;
3. conversion des unités (K → °C, J m⁻² → W m⁻², m → mm/jour, Pa → hPa) ;
4. distributions et corrélations.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

In [ ]:
from src import config as C
from src import preprocessing as pp

ds = pp.open_raw()
ds

## 1. Réduction spatiale 4×4

`coarsen` moyenne les blocs de 4×4 points : la résolution passe de 0.1° à 0.4°. La moyenne ignore les NaN, donc un bloc à cheval sur la côte garde ses points terrestres ; un bloc entièrement en mer reste NaN.

In [ ]:
ds_coarse = pp.coarsen(ds)
print(f"{float(ds['latitude'].diff('latitude').mean()):.2f}° -> "
      f"{float(ds_coarse['latitude'].diff('latitude').mean()):.2f}°")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (titre, d) in zip(axes, [("0.1° (natif)", ds), ("0.4° (4×4)", ds_coarse)]):
    d["t2m"].isel(time=6).plot(ax=ax, cmap="inferno", add_colorbar=True)
    ax.set_title(f"t2m juillet — {titre}")
plt.tight_layout();

## 2. Valeurs manquantes

Un point dont `t2m` est NaN sur toute la période est hors du domaine ERA5-Land : mer, ou hors de la zone couverte.

In [ ]:
long = pp.to_long(ds_coarse)
rep = pp.missing_report(long)
print(rep["type"].value_counts().to_string())
rep.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for t, sub in rep.groupby("type"):
    ax.scatter(sub["longitude"], sub["latitude"], s=16, label=t)
ax.set(xlabel="longitude", ylabel="latitude", title="Points de grille retenus")
ax.legend();

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
part = long.set_index("time")[[v for v in C.SHORT_NAMES if v in long]].isna().mean(axis=1)
part.groupby(level=0).mean().plot(ax=ax, title="Part de valeurs manquantes par mois")
ax.set_ylabel("fraction");

## 3. Nettoyage et conversion des unités

In [ ]:
clean = pp.add_calendar(pp.convert_units(pp.drop_sea_points(long)))
print(f"{len(long):,} lignes -> {len(clean):,} lignes retenues "
      f"({clean.groupby(['latitude','longitude']).ngroups:,} points de grille)")
pp.describe(clean).round(2)

## 4. Distributions

In [ ]:
cols = [v for v in C.SHORT_NAMES if v in clean]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, v in zip(axes.ravel(), cols):
    sns.histplot(clean[v], bins=60, ax=ax)
    ax.set_title(v)
    ax.set_xlabel("")
plt.tight_layout();

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=clean, x="month", y="t2m", ax=axes[0]).set(title="Cycle saisonnier — t2m (°C)")
sns.boxplot(data=clean, x="month", y="str", ax=axes[1]).set(title="Cycle saisonnier — rayonnement thermique net (W m⁻²)")
plt.tight_layout();

## 5. Corrélations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(pp.correlations(clean), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Corrélations de Pearson");

In [ ]:
annuel = clean.groupby("year")["t2m"].mean()
fig, ax = plt.subplots(figsize=(9, 4))
annuel.plot(marker="o", ax=ax, title="Température moyenne annuelle sur l'Algérie (°C)")
pente = np.polyfit(annuel.index, annuel.values, 1)[0]
ax.plot(annuel.index, np.poly1d(np.polyfit(annuel.index, annuel.values, 1))(annuel.index),
        "--", color="crimson", label=f"tendance : {pente:+.3f} °C/an")
ax.legend();

## Sauvegarde

In [ ]:
clean.to_parquet(C.FILE_CLEAN, index=False)
print("écrit :", C.FILE_CLEAN)